# FFTW C2C Numba Sequential

- https://github.com/numba/numba/issues/5864
- https://numba.pydata.org/numba-doc/latest/user/withobjmode.html
- https://numba.pydata.org/numba-doc/dev/reference/types.html

In [41]:
#=-----------------------------------------------------------------------=#

In [1]:
import numba
print(numba.__version__)

0.51.2


In [2]:
import pyfftw
print(pyfftw.__version__)

0.12.0


In [3]:
import mpi4py_fft
print(mpi4py_fft.__version__)

2.0.3


In [8]:
import numpy as np, pyfftw as pf, time as tm
from numba import njit, objmode

@njit
def ff(NN) :
    with objmode(a = 'complex128[:,:,:]', t0 = 'double') :  # annotate return type
        t0 = tm.time()    # <--- time measurement
        a = pf.empty_aligned( (NN, NN, NN), dtype=np.complex128 )
    for i in range(NN) :
        for j in range(NN) :
            for k in range(NN) :
                a[i, j, k] = (i*NN**2 + j*NN  + k + 1)*1E-6
    with objmode(b = 'complex128[:,:,:]') :  # annotate return type
        b = pf.interfaces.numpy_fft.fftn(a)
    s = np.sum(b)
    with objmode() :
        t1 = tm.time()    # <--- time measurement
        print('S: {:.2f}    T: {:.4f} s'.format(s, t1-t0))

ff(500)

S: 125.00-0.00j    T: 11.7657 s


In [9]:
%%writefile bc2cs.py
import numpy as np, pyfftw as pf, time as tm
from numba import njit, objmode

@njit
def ff(NN) :
    with objmode(a = 'complex128[:,:,:]', t0 = 'double') :  # annotate return type
        t0 = tm.time()    # <--- time measurement
        a = pf.empty_aligned( (NN, NN, NN), dtype=np.complex128 )
    for i in range(NN) :
        for j in range(NN) :
            for k in range(NN) :
                a[i, j, k] = (i*NN**2 + j*NN  + k + 1)*1E-6
    with objmode(b = 'complex128[:,:,:]') :  # annotate return type
        b = pf.interfaces.numpy_fft.fftn(a)
    s = np.sum(b)
    with objmode() :
        t1 = tm.time()    # <--- time measurement
        print('S: {:.2f}    T: {:.4f} s'.format(s, t1-t0))

ff(500)

Overwriting bc2cs.py


In [10]:
! cp bc2cs.py /scratch${PWD#"/prj"}

In [1]:
%%writefile bc2cs.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu_dev  : 20 min.,  1-4  nodes, 1/1   tasks
#   cpu_small: 72 hours, 1-20 nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name bc2cs       # Job name
#SBATCH --time=00:02:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes

echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd ~ && cd /scratch${PWD#"/prj"}
module load  anaconda3/2020.11  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source  /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate  ./env2
cd fft
                                              
# Executable
EXEC="python bc2cs.py"

# Start
echo '$ srun -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting bc2cs.srm


In [24]:
! sbatch --partition=cpu_dev bc2cs.srm

Submitted batch job 873925


In [25]:
! squeue -n bc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            873925    cpu_dev   R  0:02     1   24


In [28]:
! squeue -n bc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [29]:
! cat /scratch${PWD#"/prj"}/slurm-873925.out

- Job ID: 873925
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1243
$ srun -n 1 python bc2cs.py
-- output -----------------------------
S: 125.00-0.00j    T: 12.9984 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [30]:
! sbatch  bc2cs.srm
! sbatch  bc2cs.srm

Submitted batch job 873927
Submitted batch job 873928


In [31]:
! squeue -n bc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            873927  cpu_small  PD  0:00     1    1
            873928  cpu_small  PD  0:00     1    1


In [1]:
! squeue -n bc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-873927.out
! cat /scratch${PWD#"/prj"}/slurm-873928.out

- Job ID: 873927
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1325
$ srun -n 1 python bc2cs.py
-- output -----------------------------
srun: Job step aborted: Waiting up to 302 seconds for job step to finish.
slurmstepd: error: *** JOB 873927 ON sdumont1325 CANCELLED AT 2021-03-01T08:12:06 DUE TO TIME LIMIT ***
slurmstepd: error: *** STEP 873927.0 ON sdumont1325 CANCELLED AT 2021-03-01T08:12:06 DUE TO TIME LIMIT ***
srun: got SIGCONT
srun: forcing job termination
srun: error: sdumont1325: task 0: Terminated
- Job ID: 873928
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1325
$ srun -n 1 python bc2cs.py
-- output -----------------------------
S: 125.00-0.00j    T: 13.0159 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [2]:
! sbatch --partition=cpu_dev  bc2cs.srm

Submitted batch job 876112


In [3]:
! squeue -n bc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            876112    cpu_dev   R  0:01     1   24


In [6]:
! squeue -n bc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [7]:
! cat /scratch${PWD#"/prj"}/slurm-876112.out

- Job ID: 876112
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1247
$ srun -n 1 python bc2cs.py
-- output -----------------------------
S: 125.00-0.00j    T: 12.9958 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
